# 🦙 LLaMA 3.1 8B Instruct (Native FP8) — High-Speed OpenAI API Server on Google Colab (L4 GPU)

Runs **Meta Llama 3.1 8B Instruct FP8** (`neuralmagic/Meta-Llama-3.1-8B-Instruct-FP8`, ~8.5 GB VRAM) using **vLLM's native FP8 Tensor Core acceleration** with `--enforce-eager`.

Exposes standard **OpenAI-compatible** endpoints (`/v1/chat/completions`, `/v1/models`), acting as a remote high-throughput backend for **Impenetra NeMo Guardrails** (`NEMO_MAIN_MODEL_API_BASE`) and **Garak** red-team probes.

Public tunnel is provided by **Cloudflare Tunnel** (`cloudflared`) — no account, token, or rate limits.

## API Endpoints Overview

| Endpoint | Format | Primary Use Case |
|---|---|---|
| `POST /v1/chat/completions` | OpenAI format | NeMo Guardrails, Garak generators, LangChain, Tool Calling |
| `GET /v1/models` | OpenAI format | Model discovery and verification for OpenAI client libraries |
| `GET /health` | JSON | Server readiness & GPU health probe |

> **Runtime**: Set *Runtime → Change runtime type → **L4 GPU*** (24 GB VRAM).
> **Memory**: FP8 requires only ~8.5 GB VRAM (leaves 15 GB free!).
> **Speed**: Runs on NVIDIA L4's 4th-gen FP8 Tensor Cores (>100 tokens/sec) with zero JIT lag.

## Step 1 — Verify GPU & Install Dependencies

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ CUDA GPU not detected! Go to Runtime → Change runtime type → L4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
assert vram_gb >= 12.0, f"⚠️ Requires at least 12 GB VRAM. Detected: {vram_gb:.1f} GB."

# Install vLLM and huggingface_hub
!pip install -q vllm huggingface_hub

## Step 2 — Authenticate with Hugging Face

This model is governed under Meta's Llama 3.1 Community License.
Make sure you have accepted the licence at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct, then paste your **read** token.

In [ ]:
from huggingface_hub import login

# Paste your Hugging Face read token when prompted
login()

## Step 3 — Download cloudflared (public tunnel, no account needed)

[Cloudflare Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/)
creates a public HTTPS URL with **no account, no token, and no rate limit** —
ideal for high-throughput garak probe runs (`--parallel 32`).

In [ ]:
import subprocess

result = subprocess.run(
    ["wget", "-q",
     "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
     "-O", "cloudflared"],
    capture_output=True, text=True
)
subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("✅ cloudflared ready.")

## Step 4 — Launch vLLM Native OpenAI API Server & Cloudflare Tunnel

Starts vLLM's built-in production OpenAI API server (`vllm.entrypoints.openai.api_server`).

### Key Advantages:
* **Continuous Batching**: Concurrent requests (up to `max_num_seqs=64`) are continuously scheduled into parallel GPU forward passes.
* **CUDA Graphs**: Pre-captured execution graphs eliminate CPU launch overhead across concurrent worker threads.
* **Native Chat Templates**: Automatically applies Meta Llama 3.1's official tokenizer chat template with zero manual string hacking.
* **Standard OpenAI Protocol**: Full native support for `/v1/chat/completions`, streaming, token counts, and `/v1/models`.

In [ ]:
import os
import re
import subprocess
import threading
import time
import requests

PORT = 8000
PUBLIC_URL = None
LOG_FILE = "vllm_server.log"

# FP8 model cuts VRAM to 8.5 GB and accelerates L4 Ada Lovelace Tensor Cores
FP8_MODEL_ID = "RedHatAI/Meta-Llama-3.1-8B-Instruct-FP8"
PUBLIC_MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# ── 1. Launch vLLM Native OpenAI API Server with Native FP8
print(f"🚀 Starting vLLM OpenAI Server for {FP8_MODEL_ID} on port {PORT}...")

vllm_cmd = [
    "vllm", "serve", FP8_MODEL_ID,
    "--served-model-name", PUBLIC_MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.75",  # ~9 GB VRAM allocation (leaves 15 GB free on L4)
    "--max-num-seqs", "32",
    "--enforce-eager",                   # bypasses ~60s Triton / CUDA graph compilation freeze
    "--enable-auto-tool-choice",         # native tool calling
    "--tool-call-parser", "llama3_json", # native official Llama 3.1 tool call parser
    "--trust-remote-code",
]

log_f = open(LOG_FILE, "w")
server_proc = subprocess.Popen(vllm_cmd, stdout=log_f, stderr=subprocess.STDOUT, text=True)

In [ ]:
print("⏳ Waiting for vLLM to download FP8 weights (~8.5 GB) and initialize (~1-2 mins)...")
server_ready = False
for i in range(120):
    if server_proc.poll() is not None:
        log_f.close()
        with open(LOG_FILE, "r") as lf:
            logs = lf.read()
        print("❌ vLLM server crashed on startup! Last logs:")
        print(logs[-2000:])
        raise RuntimeError(f"vLLM exited with code {server_proc.returncode}")

    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=2)
        if r.status_code == 200:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not server_ready:
    log_f.close()
    with open(LOG_FILE, "r") as lf:
        print(lf.read()[-2000:])
    raise RuntimeError("vLLM server failed to become ready within timeout window.")

print("✅ vLLM OpenAI Server is online with Native FP8!")

# ── 2. Start Cloudflare Tunnel
def _run_cloudflared():
    global PUBLIC_URL
    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    url_re = re.compile(r"https://[a-z0-9\-]+\.trycloudflare\.com")
    for line in proc.stdout:
        match = url_re.search(line)
        if match and PUBLIC_URL is None:
            PUBLIC_URL = match.group(0)

threading.Thread(target=_run_cloudflared, daemon=True).start()

for _ in range(40):
    if PUBLIC_URL:
        break
    time.sleep(0.5)

if not PUBLIC_URL:
    raise RuntimeError("cloudflared did not produce a public URL in time.")

print("\n" + "="*65)
print(f"🌐 Public LLaMA 3.1 8B (FP8) API URL : {PUBLIC_URL}")
print("="*65)
print(f"  {PUBLIC_URL}/v1/chat/completions              ← OpenAI chat completions endpoint")
print(f"  {PUBLIC_URL}/v1/models                        ← OpenAI models list")
print(f"  {PUBLIC_URL}/health                           ← Health & readiness check")
print(f"  {PUBLIC_URL}/docs                             ← Swagger UI")
print()
print("► NeMo Guardrails integration:")
print(f"    export NEMO_MAIN_MODEL_API_BASE='{PUBLIC_URL}'")
print(f"    bash impenetra/experiments/nemo_guardrails/run.sh '{PUBLIC_URL}'")
print()
print("► Garak / OpenAI API Base integration:")
print(f"    export OPENAI_API_BASE='{PUBLIC_URL}/v1'")
print(f"    garak --model_type openai --model_name {PUBLIC_MODEL_NAME} --parallel 32")

## Step 5 — Smoke-test the API

Verifies `/v1/chat/completions`, `/v1/models`, and `/health` using standard requests.

In [ ]:
import time
import requests as req

BASE = f"http://127.0.0.1:{PORT}"
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
print(f"Testing against: {BASE}\n")

# 1. Test OpenAI Text Completion
print("── 1. Testing OpenAI /v1/chat/completions ──")
t0 = time.perf_counter()
openai_payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "user", "content": "Reply with exactly: PONG"}
    ],
    "temperature": 0.0,
    "max_tokens": 32,
}
r_openai = req.post(f"{BASE}/v1/chat/completions", json=openai_payload, timeout=30)
ms = round((time.perf_counter() - t0) * 1000)
print(f"[{r_openai.status_code}] ({ms} ms):")
print(r_openai.json()["choices"][0]["message"]["content"])
print()

# 2. Test Tool Calling
print("── 2. Testing OpenAI Tool Calling ──")
tool_payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "user", "content": "What is the weather in Tokyo?"}
    ],
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather in a given city",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "The city name"}
                    },
                    "required": ["city"]
                }
            }
        }
    ]
}
r_tool = req.post(f"{BASE}/v1/chat/completions", json=tool_payload, timeout=30)
choice = r_tool.json()["choices"][0]
print("Finish reason:", choice.get("finish_reason"))
print("Tool calls:", choice["message"].get("tool_calls"))
print()

# 3. Test Models
print("── 3. Testing OpenAI /v1/models ──")
r_models = req.get(f"{BASE}/v1/models", timeout=5)
print(f"[{r_models.status_code}] Models: {[m['id'] for m in r_models.json()['data']]}")
print()

# 4. Test Health
print("── 4. Testing Health Check ──")
r_health = req.get(f"{BASE}/health", timeout=5)
print(f"[{r_health.status_code}] Health → {r_health.status_code}")


## Step 6 — Exit & Teardown

In [ ]:
from google.colab import runtime
runtime.unassign()